## 3.4 Multi Schema用法
### 3.4.1 状态类型 

`LangGraph` 支持在一个图中使用多个状态 Schema，用于区分图的外部输入、外部输出、内部共享状态以及节点间的临时状态。

常见状态类型可以分为以下几类：

- 全局状态 / 内部状态：图内部主要使用的状态，创建 `StateGraph` 时传递给 `state_schema` 参数。它通常包含图运行过程中需要读写的大部分字段。
- 输入状态：图对外接收输入时使用的状态，创建 `StateGraph` 时传递给 `input_schema` 参数。它用于约束调用图时允许传入哪些字段。
- 输出状态：图最终对外返回结果时使用的状态，创建 `StateGraph` 时传递给 `output_schema` 参数。它用于约束图运行结束后只返回哪些字段。
- 私有状态：图内部节点之间传递的临时状态，通常不作为图的输入，也不作为图的最终输出。它可以通过节点函数的入参类型注解声明，并在节点返回值中写入。

阅读 `StateGraph` 的源码，可以发现它内部维护了4个状态字段。前3个`state`在源码中有对应，除了 `context_schema`。
```python
    def __init__(
        self,
        state_schema: type[StateT],
        context_schema: type[ContextT] | None = None,
        *,
        input_schema: type[InputT] | None = None,
        output_schema: type[OutputT] | None = None,
        **kwargs: Unpack[DeprecatedKwargs],
    ) -> None:
```

### 3.4.2 State 之间的关系
#### 3.4.2.1 设计规范

这里只根据课本进行概括总结。下面的4个 `state` 更偏于设计规范，没有遵守也可能不会报错。

1. input_scheme 和 output_schema 的字段应定义为 state_schema 的子集。
2. private_state 的字段 与 state_schema 应该避免重复
3. private_state 的类型通过节点函数注入。但如果一个 schema 只出现在 -> PrivateState 这样的返回注解里，节点写入的字段会被静默丢弃。


在之前的例子里，private_state 与 state_schema 在定义上完全相同了。

In [2]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph,START,END
from langgraph.graph import StateGraph

class OverAllstate(TypedDict):
    """
    全局状态，用于存储全局变量
    基础状态类

    """
    # 注解是为了规定字段,这里表明 logs更新的方式是 add而不是替代
    # 第一个参数 list[str]: 告诉静态类型检查器（如 mypy、IDE）以及开发者：这个变量是一个字符串列表（list of strings）
    # 第二个参数 add: 定义追加的方式。langgraph内部会读取第二个参数,然后以这种方式更新 logs
    logs: Annotated[list[str], add]
    cur_id: Annotated[str, add]

# 节点实际就是一个函数
def node_1(state: OverAllstate) -> OverAllstate:
    """
    节点 1
    """
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": "node_1"
    }

#### 3.4.2.2 源码层面的约束

从底层机制角度说明 `LangGraph` 如何记录、裁剪和更新状态。

`LangGraph` 的状态并不是简单保存在一个普通字典中，而是会被拆分成多个可读写的状态字段。每个状态字段在底层通常对应一个 `Channel`。

1. 什么是 Channel？

这里的 `Channel`是指**运行时管理某个字段的值、读取和更新规则的对象**。 参照笔记和 源码，理解如下:

```text
Schema 声明字段及 Reducer
          ↓ _add_schema()
图登记各字段对应的 Channel
          ↓ 运行时
从 Channel 读取值 → 组装节点的 State
节点返回更新 → 更新对应的 Channel
```

In [3]:
# 框架解析后，会为两个字段选择对应的 Channel：
class OverAllstate(TypedDict):
    logs: Annotated[list[str], add] # -> channel类型: BinaryOperatorAggregate, 调用 add 合并列表
    cur_id: str # -> channel类型: LastValue, 调用本次更新替换旧值
    # 因此，Channel 可以理解为 一个字段的值，加上管理这个值的规则。

Channel的父类 如下，提供了如下操作

* channel.get()                    # 读取当前值
* channel.update(本轮的更新列表)     # 应用本轮更新
* channel.checkpoint()             # 提取可以保存的状态
* channel.from_checkpoint(...)     # 从已保存状态恢复

```python
class BaseChannel(Generic[Value, Update, Checkpoint], ABC):
```

2. langgraph 记录State 的 核心方法是 : `_add_schema()`

它位于 [`StateGraph._add_schema()` (line 343)](/Users/skk/Developer/langgraph_learn/.venv/lib/python3.11/site-packages/langgraph/graph/state.py:343)，主要完成四件事：
* 避免重复解析同一个 Schema：已经登记过的 Schema 会跳过。
* 解析字段注解：读取字段名、类型以及 Annotated 中的 Reducer，生成对应的 Channel 定义。
* 记录 Schema 的字段集合：保存在 self.schemas 中。
* 将字段加入图的 Channel 注册表：保存在 self.channels 中；对于已有的同名字段，还会检查更新规则是否存在冲突。

这些 Schema 中的字段共同注册到同一套图状态 Channel 中。同名字段对应同一个图状态键，Schema 决定某个入口、节点或输出使用哪些字段。

私有状态是通过 add_node() 内部的 `_add_schema()` 注册的。

当添加节点时，`LangGraph` 会根据**节点函数第一个参数的类型注解**推断该节点的输入状态类型。如果这个输入状态类型之前没有被图记录过，LangGraph 也会通过 `_add_schema()` 将其加入图中。


```python
        input_schema = input_schema or inferred_input_schema
        if input_schema is not None:
            self._add_schema(input_schema)
```

#### 3.4.2.3 通过 input_schema 和 outputput_schema 管理输入输出

In [4]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

# Define the schema for the input
class InputState(TypedDict):
    question: str

# Define the schema for the output
class OutputState(TypedDict):
    answer: str

# Define the overall schema, combining both input and output
class OverallState(InputState, OutputState):
    pass

# Define the node that processes the input and generates an answer
def answer_node(state: InputState):
    # Example answer and an extra key
    return {"answer": "bye", "question": state["question"]}

# Build the graph with input and output schemas specified
builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node(answer_node)  # Add the answer node
builder.add_edge(START, "answer_node")  # Define the starting edge
builder.add_edge("answer_node", END)  # Define the ending edge
graph = builder.compile()  # Compile the graph

# Invoke the graph with an input and print the result
print(graph.invoke({"question": "hi"}))

{'answer': 'bye'}


In [7]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

# Define the schema for the input
class InputState(TypedDict):
    question: str

# Define the schema for the output
class OutputState(TypedDict):
    answer: str

# Define the overall schema, combining both input and output
class OverallState(InputState, OutputState):
    pass

# Define the node that processes the input and generates an answer
def answer_node(state: InputState):
    # Example answer and an extra key
    return {"answer": "bye", "question": state["question"]}

# Build the graph with input and output schemas specified
builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node(answer_node)  # Add the answer node
builder.add_edge(START, "answer_node")  # Define the starting edge
builder.add_edge("answer_node", END)  # Define the ending edge
graph = builder.compile()  # Compile the graph

# Invoke the graph with an input and print the result
print(graph.invoke({"answer": "hi"}))

KeyError: 'question'

### 3.4.3 案例
下面通过一个简单案例说明四类状态的定义和使用。

In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class InputState(TypedDict):
    username: str

class OutputState(TypedDict):
    graph_output: str

class OverAllState(TypedDict):
    nickname: str
    username: str
    graph_output: str

class PrivateState(TypedDict):
    greeting: str

def node_1(state: InputState) -> OverAllState:
    # 向全局状态写入数据
    return {
        "nickname": "Dear " + state["username"]
    }

def node_2(state: OverAllState) -> PrivateState:
    # 从全局状态读取数据，写入私有状态
    return {
        "greeting": state["nickname"] + ", 早上好~"
    }

def node_3(state: PrivateState) -> OutputState:
    # 从私有状态读取数据，写入输出状态
    return {
        "graph_output": state["greeting"] + " 很高兴认识你！"
    }

builder = StateGraph(OverAllState,input_schema=InputState,output_schema=OutputState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
print(graph.invoke({"username":"小黄"}))

{'graph_output': 'Dear 小黄, 早上好~ 很高兴认识你！'}


In [14]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from rich import print as rprint
class InputState(TypedDict):
    username: str

class OutputState(TypedDict):
    graph_output: str

class OverAllState(TypedDict):
    nickname: str
    username: str
    graph_output: str

class PrivateState(TypedDict):
    greeting: str
    username: str

def node_1(state: InputState) -> OverAllState:
    # 向全局状态写入数据
    return {
        "nickname": "Dear " + state["username"]
    }

def node_2(state: OverAllState) -> PrivateState:
    # 从全局状态读取数据，写入私有状态
    return {
        "greeting": state["nickname"] + ", 早上好~",
        "username": "张三"
    }

def node_3(state: PrivateState) -> OutputState:
    # 从私有状态读取数据，写入输出状态
    return {
        "graph_output": state["greeting"] + " 很高兴认识你！"
    }

builder = StateGraph(OverAllState,input_schema=InputState,output_schema=OutputState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)
# 私有与全局重复并不会报错，但会修改全局状态
graph = builder.compile()
for chunk in graph.stream({"username": "小黄"}, stream_mode="debug"):
    rprint(chunk)
print(graph.invoke({"username":"小黄"}))


{
    'step': 1,
    'timestamp': '2026-10-14T03:56:36.090619+00:00',
    'type': 'task',
    'payload': {
        'id': '8b70e7eb-5fd9-e264-deda-2e8a5829e359',
        'name': 'node_1',
        'input': {'username': '小黄'},
        'triggers': ('branch:to:node_1',)
    }
}

{
    'step': 1,
    'timestamp': '2026-10-14T03:56:36.092938+00:00',
    'type': 'task_result',
    'payload': {
        'id': '8b70e7eb-5fd9-e264-deda-2e8a5829e359',
        'name': 'node_1',
        'error': None,
        'result': {'nickname': 'Dear 小黄'},
        'interrupts': []
    }
}

{
    'step': 2,
    'timestamp': '2026-10-14T03:56:36.093079+00:00',
    'type': 'task',
    'payload': {
        'id': 'fb75c7fb-ff7d-773b-4ee2-a78113da4d53',
        'name': 'node_2',
        'input': {'nickname': 'Dear 小黄', 'username': '小黄'},
        'triggers': ('branch:to:node_2',)
    }
}

{
    'step': 2,
    'timestamp': '2026-10-14T03:56:36.095680+00:00',
    'type': 'task_result',
    'payload': {
        'id': 'fb75c7fb-ff7d-773b-4ee2-a78113da4d53',
        'name': 'node_2',
        'error': None,
        'result': {'greeting': 'Dear 小黄, 早上好~', 'username': '张三'},
        'interrupts': []
    }
}

{
    'step': 3,
    'timestamp': '2026-10-14T03:56:36.095794+00:00',
    'type': 'task',
    'payload': {
        'id': '68667670-46ec-fb2f-a326-ef5eb03bae6c',
        'name': 'node_3',
        'input': {'greeting': 'Dear 小黄, 早上好~', 'username': '张三'},
        'triggers': ('branch:to:node_3',)
    }
}

{
    'step': 3,
    'timestamp': '2026-10-14T03:56:36.098135+00:00',
    'type': 'task_result',
    'payload': {
        'id': '68667670-46ec-fb2f-a326-ef5eb03bae6c',
        'name': 'node_3',
        'error': None,
        'result': {'graph_output': 'Dear 小黄, 早上好~ 很高兴认识你！'},
        'interrupts': []
    }
}

{'graph_output': 'Dear 小黄, 早上好~ 很高兴认识你！'}


# 总结
1. Langgraph 的 state 在底层 转为 Channel 类（以字段为单位），注册过的 state 可以通过 Channel 类进行统一管理
2. 输入输出 为全局子集，而私有最好别与全局重复。


# 问题
## 1. `context_schema` 是用于做什么的?

context_schema 描述的是“本次运行使用的配置和依赖”，框架通过 Runtime.context 把它传给节点。

例如：
- State 保存计算过程中的数据：消息、检索结果、节点生成的答案。
- Context 提供本次运行的条件：当前用户 ID、选择哪个模型、数据库连接、问候语配置。

Context 通常作为本次运行的固定配置使用。

* 为什么它不走 _add_schema()？

因为 _add_schema() 的任务是把字段注册成需要读取、更新和保存的状态 Channel。Context 使用另一条传递路径：

```text
State Schema → _add_schema() → 状态 Channel → 节点的 state

Context Schema + invoke(context=...)
                            → Runtime → 节点的 runtime.context
```

对应的源码:
```python
self.context_schema = context_schema

self._add_schema(self.state_schema)
```

这里先保存 Context 的类型定义，之后编译图时将它传给运行时。在 [`invoke` 的运行路径 (line 2939)](/Users/skk/Developer/langgraph_learn/.venv/lib/python3.11/site-packages/langgraph/pregel/main.py:2939)中，框架创建：

```python
runtime = Runtime(
    context=_coerce_context(self.context_schema, context),
    # 其他运行时信息……
)
```

* 案例代码:

In [15]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime

class State(TypedDict):
    username: str
    greeting: str

class Context(TypedDict):
    greeting_prefix: str

def greet(state: State, runtime: Runtime[Context]):
    return {
        "greeting":
            runtime.context["greeting_prefix"] + state["username"]
    }

builder = StateGraph(State, context_schema=Context)
builder.add_node(greet)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)
graph = builder.compile()

result = graph.invoke(
    {"username": "小黄"},
    context={"greeting_prefix": "你好，"},
)
print(result)

{'username': '小黄', 'greeting': '你好，小黄'}
